# Capture activation traces

Run from this notebook's directory. One dense forward per prompt records projection-input importance and saves `activation_traces.json`. Continue with [02_02_global_oracle.ipynb](02_02_global_oracle.ipynb) to save the reference, then [02_03_selector_sweep.ipynb](02_03_selector_sweep.ipynb) to evaluate selectors and [02_04_analyze_oracle.ipynb](02_04_analyze_oracle.ipynb) for the figures.

PyTorch and Transformers load the checkpoint below; an uncached checkpoint is downloaded.

In [9]:
from pathlib      import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

import json
import torch

## 1. Load the model

In [10]:
# model to be used
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"

In [11]:
# bring cuda as device
device = "cuda" if torch.cuda.is_available() else "cpu"

# bring tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)

# bring model onto device
net = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    revision=MODEL_REVISION, 
    torch_dtype=torch.bfloat16
).to(device).eval()

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 566.74it/s, Materializing param=model.norm.weight]                              


## 2. Record projection inputs

A pre-forward hook records $v_i=\operatorname{mean}_{\rm batch,token}|x_i|$ at each attention or MLP projection. Absolute values and means are computed in the activation dtype, then converted to float32. The dense forward uses the original activations.

Each dictionary in `traces` contains one importance vector, in its original channel order. `trace_id` identifies the recording, `n` counts selectable input channels, `d` counts output channels, and `row_bytes` is the number of weight bytes read for one selected input channel. The values stay beside their description, in the same dictionary.

In [12]:
traces = []

def record(layer, inputs):
    n, d = layer.in_features, layer.out_features

    values = inputs[0].detach()
    values = values.abs().reshape(-1, n).mean(0)

    traces.append({
        "trace_id"  : len(traces),
        "n"         : n,
        "d"         : d,
        "row_bytes" : d * layer.weight.element_size(),
        "values"    : values.float().cpu().tolist(),
    })

In [13]:
# needed to remove hooks after the job
hooks = []

# prompts to be used
PROMPTS = (
    "Explain why contiguous flash reads can be faster than scattered reads.",
    "Summarize the trade-off between neural-network sparsity and accuracy.",
    "Write a short Python function that computes a moving average.",
)

try:
    # register hooks on projection layers.
    for name, layer in net.named_modules():
        if (
            isinstance(layer, torch.nn.Linear)
            and ".layers." in name
            and name.endswith("_proj")
        ):
            hooks.append(layer.register_forward_pre_hook(record))

    # run one forward pass per prompt.
    with torch.inference_mode():
        for input_index, prompt in enumerate(PROMPTS):
            
            # format the user message and append the assistant response prefix.
            text = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )

            # tokenize with a 256-token limit and move tensors to the device.
            tokens = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=256,
            ).to(device)

            # run a forward pass to capture layer inputs through the hooks.
            net(**tokens, use_cache=False)

            # print progress rate
            print(f"Prompt {input_index + 1} captured")

finally:
    # remove hooks even if capture fails.
    for handle in hooks:
        handle.remove()

Prompt 1 captured
Prompt 2 captured
Prompt 3 captured


## 3. Save the traces

Write the list of trace dictionaries directly to JSON. `weight_bytes` records the weight element size used by experiment 03. Model revision and prompts are the settings at the top of this notebook; they are not copied into the data file.

In [14]:
TRACE = Path("activation_traces.json")

In [15]:
data = {
    "weight_bytes": next(net.parameters()).element_size(),
    "traces": traces,
}

TRACE.write_text(json.dumps(data, indent=2))
len(traces)

504